# Task 1 Kaggle Train Notebook (`google/gemma-4-12B-it`)

Notebook nay duoc rut gon de chi giu duong chay can thiet nhat cho `Task 1`:
- setup moi truong
- login Hugging Face
- tai `AvaMERG + ESConv`
- dump prompt
- smoke test 1 step
- train that
- kiem tra checkpoint
- upload checkpoint len Hugging Face Hub
- tuy chon dung / destroy Vast instance tu trong notebook

> Neu fail, dung lai o dung cell do. Dung chay tiep den train that neu smoke test chua qua.


In [ ]:
import os
REPO_URL = "https://github.com/QuangVoAI/multimodal-empathy-mental-health.git"
REPO_DIR = "/kaggle/working/multimodal-empathy-mental-health"
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd /kaggle/working/multimodal-empathy-mental-health
# Keep Kaggle's preinstalled torch/CUDA stack.
%pip uninstall -y datasets transformers huggingface_hub accelerate peft bitsandbytes sentencepiece tokenizers torchvision
%pip install --no-cache-dir --force-reinstall -r requirements_kaggle.txt


## Restart kernel now

Sau khi cell tren chay xong, hay **Restart Session / Restart Kernel** roi moi chay tiep.


In [ ]:
%cd /kaggle/working/multimodal-empathy-mental-health
import torch, transformers, huggingface_hub
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN"
MODEL_ID = "google/gemma-4-12B-it"
RUN_NAME = "task1_gemma12b_it_joint"
HF_MODEL_REPO_ID = "SpringWang08/multimodal-empathy-mental-health-gemma12b-task1"

# Vast auto-control flags
AUTO_STOP_ON_FAILURE = False
AUTO_STOP_AFTER_UPLOAD = True
AUTO_DESTROY_AFTER_UPLOAD = False

login(HF_TOKEN)


## Optional Vast.ai auto-stop helpers

Vast co the duoc stop tu trong instance bang CLI va instance-specific API key. Theo Vast docs, `stop` se giu du lieu nhung van tinh storage cost, con `destroy` se xoa luon instance va du lieu. Nguon: [Vast FAQ](https://console.vast.ai/faq/) va [Managing Instances](https://docs.vast.ai/documentation/instances/manage-instances).


In [ ]:
import os
import subprocess
from pathlib import Path

def setup_vast_cli():
    label = os.environ.get("VAST_CONTAINERLABEL")
    auth_keys = Path.home() / ".ssh" / "authorized_keys"
    if not label:
        print("VAST_CONTAINERLABEL not found. Skip Vast control setup.")
        return False
    if not auth_keys.exists():
        print(f"{auth_keys} not found. Skip Vast control setup.")
        return False
    try:
        subprocess.run(
            "cat ~/.ssh/authorized_keys | md5sum | awk '{print $1}' > ssh_key_hv; "
            "echo -n $VAST_CONTAINERLABEL | md5sum | awk '{print $1}' > instance_id_hv; "
            "head -c -1 -q ssh_key_hv instance_id_hv > ~/.vast_api_key",
            shell=True, check=True, executable="/bin/bash"
        )
        subprocess.run(
            "apt-get install -y wget >/dev/null && wget -q https://raw.githubusercontent.com/vast-ai/vast-python/master/vast.py -O vast && chmod +x vast",
            shell=True, check=True, executable="/bin/bash"
        )
        instance_id = label.split(".", 1)[1]
        subprocess.run(f"./vast start instance {instance_id}", shell=True, check=True, executable="/bin/bash")
        print(f"Vast CLI ready for instance {instance_id}")
        return True
    except Exception as exc:
        print(f"Vast control setup skipped: {exc}")
        return False

def vast_action(action="stop"):
    label = os.environ.get("VAST_CONTAINERLABEL")
    if not label:
        print("Not running inside Vast; skip", action)
        return
    instance_id = label.split(".", 1)[1]
    if action not in {"stop", "destroy"}:
        raise ValueError("action must be stop or destroy")
    cmd = f"./vast {action} instance {instance_id}"
    print("Running:", cmd)
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")

setup_vast_cli()


In [ ]:
from argparse import Namespace
from transformers import AutoTokenizer
from scripts.train_sft import run_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded:", type(tokenizer).__name__)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Import ok")


In [ ]:
!bash scripts/download_avamerg.sh
!bash scripts/download_esconv.sh
!mkdir -p outputs/sft outputs/eval

import json
from pathlib import Path
ava = json.loads(Path("data/raw/avamerg/train.json").read_text(encoding="utf-8"))
esc = json.loads(Path("data/raw/esconv/ESConv.json").read_text(encoding="utf-8"))
print("AvaMERG samples:", len(ava))
print("ESConv dialogues:", len(esc))
print("AvaMERG first keys:", list(ava[0].keys()))
print("ESConv first keys:", list(esc[0].keys()))


In [ ]:
base_cfg = dict(
    model_name_or_path=MODEL_ID,
    avamerg_root="data/raw/avamerg",
    avamerg_split="train",
    avamerg_text_only=True,
    esconv_json="data/raw/esconv/ESConv.json",
    output_dir="outputs/sft/debug_joint",
    max_length=1536,
    max_response_tokens=192,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=1.0,
    logging_steps=1,
    save_steps=50,
    warmup_ratio=0.03,
    max_steps=-1,
    use_lora=True,
    load_in_4bit=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    report_to="none",
    dump_example_prompts=True,
)

run_training(Namespace(**base_cfg))


In [ ]:
!sed -n "1,200p" outputs/sft/debug_joint/example_prompts.json


In [ ]:
smoke_cfg = dict(base_cfg)
smoke_cfg.update({
    "output_dir": "outputs/sft/joint_smoke",
    "dump_example_prompts": False,
    "max_steps": 1,
    "logging_steps": 1,
})

try:
    run_training(Namespace(**smoke_cfg))
except Exception:
    if AUTO_STOP_ON_FAILURE:
        print("Smoke test failed. Stopping Vast instance...")
        vast_action("stop")
    raise


In [ ]:
# Chi chay cell nay sau khi smoke test da qua.
train_cfg = dict(base_cfg)
train_cfg.update({
    "output_dir": f"outputs/sft/{RUN_NAME}",
    "dump_example_prompts": False,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    "save_steps": 100,
    "max_steps": -1,
})

try:
    run_training(Namespace(**train_cfg))
except Exception:
    if AUTO_STOP_ON_FAILURE:
        print("Training failed. Stopping Vast instance...")
        vast_action("stop")
    raise


In [ ]:
from pathlib import Path

final_dir = Path(f"outputs/sft/{RUN_NAME}/final")
print("Final checkpoint dir:", final_dir)
print("Exists:", final_dir.exists())
if final_dir.exists():
    for path in sorted(final_dir.iterdir()):
        print(path.name)


In [ ]:
# Chay cell nay de upload checkpoint/adapters len Hugging Face Hub.
from pathlib import Path
from scripts.publish_to_hub import create_repo, upload_folder

final_dir = Path(f"outputs/sft/{RUN_NAME}/final")
if not final_dir.exists():
    raise FileNotFoundError(f"Checkpoint not found: {final_dir}")

create_repo(repo_id=HF_MODEL_REPO_ID, repo_type="model", private=False, exist_ok=True)
upload_folder(
    repo_id=HF_MODEL_REPO_ID,
    repo_type="model",
    folder_path=str(final_dir),
    commit_message=f"Upload {RUN_NAME} checkpoint",
)
print(f"Uploaded {final_dir} -> https://huggingface.co/{HF_MODEL_REPO_ID}")

if AUTO_DESTROY_AFTER_UPLOAD:
    print("Destroying Vast instance after upload...")
    vast_action("destroy")
elif AUTO_STOP_AFTER_UPLOAD:
    print("Stopping Vast instance after upload...")
    vast_action("stop")


In [ ]:
# Manual Vast control (optional)
# vast_action("stop")      # keep disk, storage charges continue
# vast_action("destroy")   # delete instance and data
